# ASCII Stream Engine (General)

Notebook general para:
- Levantar el servidor para la red (broadcast / multicast / IP directa)
- Control de camara de entrada
- Modo RAW / ASCII
- Filtros en vivo

**VLC (receptores):**
- Broadcast/LAN: `udp://@0.0.0.0:1234`
- Multicast: `udp://@239.0.0.1:1234`


In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Repo root:", repo_root)


In [ ]:
import os

from ascii_stream_engine import (
    EngineConfig,
    StreamEngine,
    OpenCVCameraSource,
    AsciiRenderer,
    FfmpegUdpOutput,
    FilterPipeline,
)
from ascii_stream_engine.filters import BrightnessFilter, EdgeFilter, InvertFilter

try:
    from ascii_stream_engine.filters import DetailBoostFilter
except Exception:
    DetailBoostFilter = None

FONT_PATH = "/usr/share/fonts/truetype/freefont/FreeMono.ttf"
renderer = AsciiRenderer()
if os.path.exists(FONT_PATH):
    try:
        renderer = AsciiRenderer(font_path=FONT_PATH, font_size=8)
    except TypeError:
        from PIL import ImageFont
        renderer = AsciiRenderer(font=ImageFont.truetype(FONT_PATH, 8))

config = EngineConfig(host="127.0.0.1", port=1234, fps=30, grid_w=100, grid_h=31)
filters = FilterPipeline([])

engine = StreamEngine(
    source=OpenCVCameraSource(0),
    renderer=renderer,
    sink=FfmpegUdpOutput(),
    config=config,
    filters=filters,
)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

status = widgets.HTML()

# Network controls
network_mode = widgets.Dropdown(
    options=["Local", "Broadcast", "Multicast", "IP directa"],
    value="Local",
    description="Red",
)
host_input = widgets.Text(value="127.0.0.1", description="Host")
port_input = widgets.IntText(value=1234, description="Puerto")
apply_net_btn = widgets.Button(description="Aplicar red")

# Camera controls
camera_index = widgets.IntText(value=0, description="Camara")
apply_camera_btn = widgets.Button(description="Aplicar camara")

# Filters
detail_filter = DetailBoostFilter() if DetailBoostFilter else None
edge_filter = EdgeFilter(60, 120)
brightness_filter = BrightnessFilter()
invert_filter = InvertFilter()

FILTERS = {
    "Edges": edge_filter,
    "Brightness/Contrast": brightness_filter,
    "Invert": invert_filter,
}
if detail_filter:
    FILTERS["Detail Boost"] = detail_filter

filter_checkboxes = {
    name: widgets.Checkbox(value=False, description=name) for name in FILTERS
}

DENSE_CHARSET = " .'`^\\\",:;Il!i~+_-?][}{1)(|\\\\/tfjrxnuvczXYUJCLQ0OZmwqpdbkhao*#MW&8%B@$"

fps_slider = widgets.IntSlider(value=engine.get_config().fps, min=10, max=60, description="FPS")
grid_w_slider = widgets.IntSlider(value=engine.get_config().grid_w, min=60, max=200, description="Grid W")
grid_h_slider = widgets.IntSlider(value=engine.get_config().grid_h, min=20, max=120, description="Grid H")
charset_dd = widgets.Dropdown(
    options={"Simple": " .:-=+*#", "Medio": " .:-=+*#%@", "Denso": DENSE_CHARSET},
    value=engine.get_config().charset,
    description="Charset",
)

render_mode = widgets.RadioButtons(
    options=[("ASCII", "ascii"), ("RAW (sin ASCII)", "raw")],
    value="ascii",
    description="Modo",
)

raw_width = widgets.IntText(value=640, description="Raw W")
raw_height = widgets.IntText(value=360, description="Raw H")
raw_use_size = widgets.Checkbox(value=False, description="Usar tamaño RAW")

contrast_slider = widgets.FloatSlider(value=engine.get_config().contrast, min=0.5, max=3.0, step=0.1, description="Contraste")
brightness_slider = widgets.IntSlider(value=engine.get_config().brightness, min=-50, max=50, step=1, description="Brillo")
frame_buffer_slider = widgets.IntSlider(value=engine.get_config().frame_buffer_size, min=0, max=3, step=1, description="Buffer")
bitrate_text = widgets.Text(value=str(engine.get_config().bitrate), description="Bitrate")

apply_settings_btn = widgets.Button(description="Aplicar ajustes")
clear_filters_btn = widgets.Button(description="Quitar filtros")
start_btn = widgets.Button(description="Start")
stop_btn = widgets.Button(description="Stop")


def rebuild_filters(_=None) -> None:
    selected = [FILTERS[name] for name, cb in filter_checkboxes.items() if cb.value]
    engine.filter_pipeline.replace(selected)
    if selected:
        status.value = f"Filtros activos: {', '.join([f.name for f in selected])}"
    else:
        status.value = "Sin filtros: imagen normal (sin efectos)."


def _sync_host(change) -> None:
    mode = change["new"]
    if mode == "Local":
        host_input.value = "127.0.0.1"
    elif mode == "Broadcast":
        host_input.value = "255.255.255.255"
    elif mode == "Multicast":
        host_input.value = "239.0.0.1"
    # IP directa: no tocar


def apply_network(_=None) -> None:
    mode = network_mode.value
    host = host_input.value.strip()
    udp_broadcast = False
    if mode == "Local":
        host = "127.0.0.1"
    elif mode == "Broadcast":
        udp_broadcast = True
        if not host:
            host = "255.255.255.255"
    elif mode == "Multicast":
        if not host:
            host = "239.0.0.1"
    elif mode == "IP directa":
        if not host:
            status.value = "Falta host para IP directa."
            return

    was_running = engine.is_running
    if was_running:
        engine.stop()
    engine.update_config(host=host, port=port_input.value, udp_broadcast=udp_broadcast)
    if was_running:
        engine.start()
    status.value = f"Red aplicada: {mode} -> {host}:{port_input.value}"


def apply_camera(_=None) -> None:
    source = engine.get_source()
    if not hasattr(source, "set_camera_index"):
        status.value = "Fuente actual no soporta cambio de camara."
        return
    was_running = engine.is_running
    if was_running:
        engine.stop()
    source.set_camera_index(int(camera_index.value))
    if was_running:
        engine.start()
    status.value = f"Camara cambiada a {camera_index.value}."


def apply_settings(_=None) -> None:
    was_running = engine.is_running
    if was_running:
        engine.stop()
    raw_w = raw_width.value if raw_use_size.value else None
    raw_h = raw_height.value if raw_use_size.value else None
    engine.update_config(
        fps=fps_slider.value,
        grid_w=grid_w_slider.value,
        grid_h=grid_h_slider.value,
        charset=charset_dd.value,
        contrast=contrast_slider.value,
        brightness=brightness_slider.value,
        render_mode=render_mode.value,
        raw_width=raw_w,
        raw_height=raw_h,
        frame_buffer_size=frame_buffer_slider.value,
        bitrate=bitrate_text.value,
    )
    if was_running:
        engine.start()
    status.value = "Ajustes aplicados (puede requerir reconectar VLC)."


def clear_filters(_=None) -> None:
    for cb in filter_checkboxes.values():
        cb.value = False
    rebuild_filters()


def start_engine(_=None) -> None:
    engine.start()
    status.value = "Engine corriendo."


def stop_engine(_=None) -> None:
    engine.stop()
    status.value = "Engine detenido."


apply_net_btn.on_click(apply_network)
network_mode.observe(_sync_host, names="value")
apply_camera_btn.on_click(apply_camera)
apply_settings_btn.on_click(apply_settings)
clear_filters_btn.on_click(clear_filters)
start_btn.on_click(start_engine)
stop_btn.on_click(stop_engine)

for cb in filter_checkboxes.values():
    cb.observe(rebuild_filters, names="value")

network_box = widgets.VBox([
    widgets.HTML("<b>Servidor en red</b>"),
    network_mode,
    widgets.HBox([host_input, port_input]),
    apply_net_btn,
])

engine_box = widgets.VBox([
    widgets.HBox([start_btn, stop_btn]),
    widgets.HBox([camera_index, apply_camera_btn]),
    status,
])

filters_box = widgets.VBox(
    [widgets.HTML("<b>Filtros (antes de ASCII)</b>")] +
    list(filter_checkboxes.values()) +
    [clear_filters_btn]
)

settings_box = widgets.VBox(
    [
        widgets.HTML("<b>ASCII / RAW</b>"),
        fps_slider,
        grid_w_slider,
        grid_h_slider,
        charset_dd,
        contrast_slider,
        brightness_slider,
        frame_buffer_slider,
        bitrate_text,
        render_mode,
        raw_use_size,
        widgets.HBox([raw_width, raw_height]),
        apply_settings_btn,
    ]
)

tabs = widgets.Tab(children=[network_box, engine_box, filters_box, settings_box])
tabs.set_title(0, "Red")
tabs.set_title(1, "Engine")
tabs.set_title(2, "Filtros")
tabs.set_title(3, "ASCII/RAW")

rebuild_filters()
display(tabs)
